# RQ3: Link Prediction Reliability Evaluation

This notebook evaluates the quality of predicted paper-to-paper links generated by the multi-agent ingestion pipeline against the expert-annotated citation graph in Emile's vault.

It compares the links on two configurations:
1. **Full Corpus (N=40)**: Evaluating predicted links against all paper-to-paper links in the gold vault.
2. **Intersection Subgraph (N=22)**: Evaluating links strictly between the papers that exist in both the expert vault and our corpus.

### Metrics Calculated:
- **Loose Match** (pair only, ignores relationship type): Precision, Recall, F1-score
- **Strict Match** (pair + relation type must match): Precision, Recall, F1-score
- **Relation Type Accuracy**: Accuracy of predicting the correct type, given that the link pair was correctly identified.

In [9]:
import json
import re
from pathlib import Path
import pandas as pd

# Define paths
notes_dir = Path("data/generated_notes")
gold_links_path = Path("data/gold_labels/link_gold.json")
emile_ids_path = Path("data/gold_labels/emile_vault_arxiv_ids.json")
output_path = Path("data/results/link_eval.json")

In [10]:
# 1. Load data
with open(gold_links_path, encoding="utf-8") as f:
    gold_links = json.load(f)
with open(emile_ids_path, encoding="utf-8") as f:
    emile_ids = json.load(f)
    
# Standardize expert arXiv ID map keys (lowercase)
emile_title_to_arxiv = {title.lower().strip(): arxiv_id for title, arxiv_id in emile_ids.items()}

print(f"Loaded {len(gold_links)} gold links from {gold_links_path}")
print(f"Loaded {len(emile_ids)} expert paper arXiv ID mappings")

Loaded 326 gold links from data\gold_labels\link_gold.json
Loaded 22 expert paper arXiv ID mappings


In [11]:
# 2. Build title map from generated notes
# This maps the exact titles used by our agent (in lowercase) to arXiv IDs
title_to_arxiv = {}
for note_path in notes_dir.glob("*.md"):
    arxiv_id = note_path.stem
    try:
        content = note_path.read_text(encoding="utf-8")
    except Exception:
        continue
    title_match = re.search(r'^title:\s*"(.*?)"', content, re.MULTILINE)
    if not title_match:
        title_match = re.search(r'^title:\s*(.*?)$', content, re.MULTILINE)
    if title_match:
        title = title_match.group(1).strip().strip('"').strip("'")
        title_to_arxiv[title.lower()] = arxiv_id
        
# Add Emile's titles to the resolver as well, to handle minor naming mismatches
for title, arxiv_id in emile_ids.items():
    if title.lower() not in title_to_arxiv:
        title_to_arxiv[title.lower()] = arxiv_id

print(f"Title map size: {len(title_to_arxiv)} unique titles mapped to arXiv IDs")

Title map size: 54 unique titles mapped to arXiv IDs


In [12]:
# 3. Parse predicted links from the generated notes
predicted_links = []

for note_path in notes_dir.glob("*.md"):
    source_id = note_path.stem  # arXiv ID
    try:
        content = note_path.read_text(encoding="utf-8")
    except Exception as e:
        print(f"Error reading {note_path}: {e}")
        continue
        
    parts = content.split("## Generated Links")
    if len(parts) < 2:
        continue
        
    links_section = parts[1]
    lines = links_section.strip().split("\n")
    
    for line in lines:
        line = line.strip()
        if not line.startswith("-"):
            continue
            
        # Match pattern: - **[[Target Title]]** (type): justification
        match = re.search(r"-\s*\*\*\[\[(.*?)\]\]\*\*\s*\((.*?)\)", line)
        if match:
            target_title = match.group(1).strip()
            link_type = match.group(2).strip()
            
            target_id = title_to_arxiv.get(target_title.lower())
            
            if target_id:
                predicted_links.append({
                    "source": source_id,
                    "target": target_id,
                    "link_type": link_type
                })

print(f"Total parsed predicted links: {len(predicted_links)}")

Total parsed predicted links: 189


In [13]:
# 4. Filter and map gold links to paper-to-paper links
gold_paper_links = []

for gl in gold_links:
    # Check if they are paper-to-paper
    if gl.get("source_type") != "paper" or gl.get("target_type") != "paper":
        continue
        
    src_title = gl["source"]
    tgt_title = gl["target"]
    
    # Resolve source and target to arXiv IDs
    src_id = emile_title_to_arxiv.get(src_title.lower().strip()) or title_to_arxiv.get(src_title.lower().strip())
    tgt_id = emile_title_to_arxiv.get(tgt_title.lower().strip()) or title_to_arxiv.get(tgt_title.lower().strip())
    
    if src_id and tgt_id:
        gold_paper_links.append({
            "source": src_id,
            "target": tgt_id,
            "link_type": gl.get("link_type", "cites")
        })

print(f"Total gold paper-to-paper links resolved: {len(gold_paper_links)}")

Total gold paper-to-paper links resolved: 1


In [14]:
# 5. Define corpus papers and intersection papers
corpus_arxiv_ids = {p.stem for p in notes_dir.glob("*.md")}
expert_arxiv_ids = set(emile_ids.values())
intersecting_arxiv_ids = corpus_arxiv_ids & expert_arxiv_ids

# 6. Evaluation Configuration A: Full Corpus Evaluation
pred_set_loose = {(l["source"], l["target"]) for l in predicted_links}
pred_set_strict = {(l["source"], l["target"], l["link_type"]) for l in predicted_links}

gold_set_loose = {(l["source"], l["target"]) for l in gold_paper_links if l["source"] in corpus_arxiv_ids and l["target"] in corpus_arxiv_ids}
gold_set_strict = {(l["source"], l["target"], l["link_type"]) for l in gold_paper_links if l["source"] in corpus_arxiv_ids and l["target"] in corpus_arxiv_ids}

# Configuration B: Intersection Subgraph Evaluation
pred_intersect_loose = {(src, tgt) for src, tgt in pred_set_loose if src in intersecting_arxiv_ids and tgt in intersecting_arxiv_ids}
pred_intersect_strict = {(src, tgt, ltype) for src, tgt, ltype in pred_set_strict if src in intersecting_arxiv_ids and tgt in intersecting_arxiv_ids}

gold_intersect_loose = {(src, tgt) for src, tgt in gold_set_loose if src in intersecting_arxiv_ids and tgt in intersecting_arxiv_ids}
gold_intersect_strict = {(src, tgt, ltype) for src, tgt, ltype in gold_set_strict if src in intersecting_arxiv_ids and tgt in intersecting_arxiv_ids}

# Helper function to compute metrics
def compute_metrics(preds, gold, preds_strict, gold_strict):
    tp_loose = len(preds & gold)
    p_loose = tp_loose / len(preds) if preds else 0.0
    r_loose = tp_loose / len(gold) if gold else 0.0
    f1_loose = 2 * p_loose * r_loose / (p_loose + r_loose) if p_loose + r_loose > 0 else 0.0
    
    tp_strict = len(preds_strict & gold_strict)
    p_strict = tp_strict / len(preds_strict) if preds_strict else 0.0
    r_strict = tp_strict / len(gold_strict) if gold_strict else 0.0
    f1_strict = 2 * p_strict * r_strict / (p_strict + r_strict) if p_strict + r_strict > 0 else 0.0
    
    type_acc = tp_strict / tp_loose if tp_loose > 0 else 0.0
    
    return {
        "n_predicted": len(preds),
        "n_gold": len(gold),
        "loose_match": {
            "precision": p_loose,
            "recall": r_loose,
            "f1": f1_loose,
            "tp": tp_loose
        },
        "strict_match": {
            "precision": p_strict,
            "recall": r_strict,
            "f1": f1_strict,
            "tp": tp_strict
        },
        "relation_type_accuracy": type_acc
    }

metrics_full = compute_metrics(pred_set_loose, gold_set_loose, pred_set_strict, gold_set_strict)
metrics_intersect = compute_metrics(pred_intersect_loose, gold_intersect_loose, pred_intersect_strict, gold_intersect_strict)

In [15]:
# 7. Format and display results as a pandas DataFrame
df_data = {
    "Metric": [
        "Loose Precision (Pair Only)", 
        "Loose Recall (Pair Only)", 
        "Loose F1-score (Pair Only)", 
        "Strict Precision (Pair + Type)", 
        "Strict Recall (Pair + Type)", 
        "Strict F1-score (Pair + Type)",
        "Relation Type Accuracy",
        "TP Links Count",
        "Total Gold Links in scope",
        "Total Predicted Links in scope"
    ],
    "Full Corpus Evaluation (N=40)": [
        f"{metrics_full['loose_match']['precision']:.5f}",
        f"{metrics_full['loose_match']['recall']:.5f}",
        f"{metrics_full['loose_match']['f1']:.5f}",
        f"{metrics_full['strict_match']['precision']:.5f}",
        f"{metrics_full['strict_match']['recall']:.5f}",
        f"{metrics_full['strict_match']['f1']:.5f}",
        f"{metrics_full['relation_type_accuracy']:.5f}",
        f"{metrics_full['loose_match']['tp']} (loose) / {metrics_full['strict_match']['tp']} (strict)",
        metrics_full['n_gold'],
        metrics_full['n_predicted']
    ],
    "Intersection Subgraph (N=22)": [
        f"{metrics_intersect['loose_match']['precision']:.5f}",
        f"{metrics_intersect['loose_match']['recall']:.5f}",
        f"{metrics_intersect['loose_match']['f1']:.5f}",
        f"{metrics_intersect['strict_match']['precision']:.5f}",
        f"{metrics_intersect['strict_match']['recall']:.5f}",
        f"{metrics_intersect['strict_match']['f1']:.5f}",
        f"{metrics_intersect['relation_type_accuracy']:.5f}",
        f"{metrics_intersect['loose_match']['tp']} (loose) / {metrics_intersect['strict_match']['tp']} (strict)",
        metrics_intersect['n_gold'],
        metrics_intersect['n_predicted']
    ]
}

results_df = pd.DataFrame(df_data)
results_df

,Metric,Full Corpus Evaluation (N=40),Intersection Subgraph (N=22)
0,Loose Precision (Pair Only),0.00529,0.01124
1,Loose Recall (Pair Only),1.00000,1.00000
2,Loose F1-score (Pair Only),0.01053,0.02222
3,Strict Precision (Pair + Type),0.00000,0.00000
4,Strict Recall (Pair + Type),0.00000,0.00000
5,Strict F1-score (Pair + Type),0.00000,0.00000
6,Relation Type Accuracy,0.00000,0.00000
7,TP Links Count,1 (loose) / 0 (strict),1 (loose) / 0 (strict)
8,Total Gold Links in scope,1,1
9,Total Predicted Links in scope,189,89


In [16]:
# 8. Export results to JSON
results = {
    "evaluation_scope": "RQ3 - Link Prediction Reliability",
    "parameters": {
        "min_link_confidence_threshold": 0.6
    },
    "full_corpus_metrics": metrics_full,
    "intersection_subgraph_metrics": metrics_intersect,
    "raw_gold_paper_to_paper_links": [
        {"source": l["source"], "target": l["target"], "link_type": l["link_type"]} 
        for l in gold_paper_links
    ]
}

output_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
    
print(f"Results successfully saved to {output_path}")

Results successfully saved to data\results\link_eval.json
